In [2]:
import pandas as pd

df = pd.read_csv("../dataset/IMDB Dataset.csv")

print(df.head())
print(df.shape)
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
import re

# Convert positive/negative into 1 and 0
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['review'] = df['review'].apply(clean_text)

print(df.head())

                                              review  sentiment
0  one of the other reviewers has mentioned that ...          1
1  a wonderful little production br br the filmin...          1
2  i thought this was a wonderful way to spend ti...          1
3  basically theres a family where a little boy j...          0
4  petter matteis love in the time of money is a ...          1


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Input and Output
X = df['review']
y = df['sentiment']

# Create tokenizer
tokenizer = Tokenizer(num_words=5000)

# Learn words from reviews
tokenizer.fit_on_texts(X)

# Convert text into numbers
X_seq = tokenizer.texts_to_sequences(X)

# Make all reviews same length
X_pad = pad_sequences(X_seq, maxlen=200)

print(X_pad.shape)

(50000, 200)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(40000, 200)
(10000, 200)


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 94s 144ms/step - accuracy: 0.8031 - loss: 0.4330 - val_accuracy: 0.8635 - val_loss: 0.3256
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 69s 110ms/step - accuracy: 0.8847 - loss: 0.2906 - val_accuracy: 0.8849 - val_loss: 0.2839
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 81s 109ms/step - accuracy: 0.9041 - loss: 0.2419 - val_accuracy: 0.8737 - val_loss: 0.3138
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 70s 112ms/step - accuracy: 0.9190 - loss: 0.2094 - val_accuracy: 0.8710 - val_loss: 0.3472
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 96s 154ms/step - accuracy: 0.9293 - loss: 0.1824 - val_accuracy: 0.8813 - val_loss: 0.2969


In [9]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.8813 - loss: 0.2969
Accuracy: 0.8812999725341797


In [12]:
model.save("../models/sentiment_model.keras")

import pickle

with open("../models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)